# B03 — Parser Quality Eval (via `/parser` API)

Sends each premise group's NL premises to `/parser` and inspects the FOL output.

- **Input:** `Logic_Based_Educational_Queries.json` — full labelled split.
- **Pipeline:** premises-NL → `/parser` → FOL ASTs.
- **Scoring:** verified rate, issue type breakdown, parse coverage.


In [ ]:
import json, time, asyncio, statistics
from pathlib import Path
from collections import Counter

import httpx

# --- endpoint -------------------------------------------------------------
API_BASE    = "https://api.iamphuckhang.dev"
PARSER_URL  = f"{API_BASE}/parser"

# --- run size -------------------------------------------------------------
N_GROUPS    = 5        # how many premise groups to eval (None = all)
CONCURRENCY = 8        # parallel in-flight requests
TIMEOUT     = 120.0    # per-request seconds

# --- locate dataset -------------------------------------------------------
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src/exact/datasets/exact").exists():
    ROOT = ROOT.parent
DATA = ROOT / "src/exact/datasets/exact"
assert DATA.exists(), f"dataset dir not found from {Path.cwd()}"
print("dataset :", DATA)
print("endpoint:", PARSER_URL)


In [ ]:
raw = json.load(open(DATA / "Logic_Based_Educational_Queries.json"))

# Each group has premises-NL + questions + answers.
# We only care about premises for parser quality.
groups = [
    {
        "id": f"group_{i:04d}",
        "premises": g["premises-NL"],
    }
    for i, g in enumerate(raw)
]

print(f"{len(groups)} groups total")
print(f"Example group: {groups[0]['id']}")
for p in groups[0]["premises"]:
    print(f"  {p}")


In [ ]:
async def call_parser(client, sem, group):
    async with sem:
        t0 = time.perf_counter()
        err, parsed_premises, verified, issues, renames = None, [], False, [], []
        try:
            r = await client.post(
                PARSER_URL,
                json={"premises": group["premises"]},
                timeout=TIMEOUT,
            )
            r.raise_for_status()
            body = r.json()
            parsed_premises = body.get("premises", [])
            verified        = body.get("verified", False)
            issues          = body.get("issues", [])
            renames         = body.get("renames", [])
        except Exception as e:
            err = repr(e)
        dt = time.perf_counter() - t0

    return {
        "id":       group["id"],
        "premises_nl":  group["premises"],
        "premises_fol": parsed_premises,
        "verified":     verified,
        "issues":       issues,
        "renames":      renames,
        "latency":      dt,
        "error":        err,
    }


async def run_eval(groups):
    sem  = asyncio.Semaphore(CONCURRENCY)
    done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(g):
            nonlocal done
            res = await call_parser(client, sem, g)
            done += 1
            if done % 5 == 0 or done == len(groups):
                print(f"  {done}/{len(groups)}", end="\r")
            return res
        return await asyncio.gather(*(wrapped(g) for g in groups))


In [ ]:
subset = groups[:N_GROUPS] if N_GROUPS else groups
print(f"Evaluating {len(subset)} groups at concurrency {CONCURRENCY}...")
t0 = time.perf_counter()
results = await run_eval(subset)
wall = time.perf_counter() - t0

errors  = [r for r in results if r["error"]]
success = [r for r in results if not r["error"]]
print(f"\nSuccess : {len(success)}/{len(results)}")
print(f"Errors  : {len(errors)}")
print(f"Wall    : {wall:.1f}s")


In [ ]:
# --- Error log ---
if errors:
    print("=== Errors ===")
    for r in errors:
        print(f"  [{r['id']}] {r['error']}")
    print()

# --- NL → FOL per group ---
print("=== Premises NL → FOL ===")
for r in success:
    flag = "✓" if r["verified"] else "✗"
    print(f"\n{flag} {r['id']}  ({len(r['premises_nl'])} premises, {r['latency']:.1f}s)")
    fol_by_text = {p["original_text"]: p["fol"] for p in r["premises_fol"]}
    for nl in r["premises_nl"]:
        fol = fol_by_text.get(nl, "<not parsed>")
        print(f"  NL : {nl}")
        print(f"  FOL: {fol}")
    if r["issues"]:
        for issue in r["issues"]:
            print(f"  !! {issue}")
    if r["renames"]:
        print(f"  renames: {r['renames']}")


In [ ]:
# --- Quality metrics ---
n = len(success)
if n == 0:
    print("No successful results to report.")
else:
    verified_count   = sum(r["verified"] for r in success)
    has_issues_count = sum(bool(r["issues"]) for r in success)
    has_renames_count= sum(bool(r["renames"]) for r in success)

    # Count how many premises actually got a FOL parse
    total_nl  = sum(len(r["premises_nl"]) for r in success)
    total_fol = sum(len(r["premises_fol"]) for r in success)

    print("=== Quality Metrics ===")
    print(f"  Groups evaluated : {n}")
    print(f"  Verified         : {verified_count}/{n}  ({verified_count/n:.1%})")
    print(f"  Has issues       : {has_issues_count}/{n}  ({has_issues_count/n:.1%})")
    print(f"  Has renames      : {has_renames_count}/{n}  ({has_renames_count/n:.1%})")
    print(f"  Premises NL      : {total_nl}")
    print(f"  Premises parsed  : {total_fol}  ({total_fol/total_nl:.1%} coverage)")

    # Issue type breakdown
    issue_types: Counter = Counter()
    for r in success:
        for issue in r["issues"]:
            tag = issue.split(":")[0].strip()
            issue_types[tag] += 1

    if issue_types:
        print("\n=== Issue Type Breakdown ===")
        for tag, cnt in issue_types.most_common():
            print(f"  {tag:45s}: {cnt}")

    # Latency
    lat = [r["latency"] for r in success]
    print(f"\n=== Latency ===")
    print(f"  mean={statistics.mean(lat):.2f}s  p50={statistics.median(lat):.2f}s  max={max(lat):.2f}s")


## Notes
- Set `N_GROUPS = None` to run all groups.
- `verified=True` means no blocking schema diagnostics were raised.
- Issue tags: `ARITY_DRIFT`, `ONLY_IF_DIRECTION_CHECK`, `NUMERIC_CONSTRAINT_LOST`, `TEMPORAL_CONSTRAINT_LOST`, `UNSUPPORTED_MODAL_NOT_NECESSARILY`.
- `renames` shows predicates that were collapsed to a canonical name during schema build.
- To inspect one result: `next(r for r in results if r['id'] == 'group_0000')`.
